# Hotel Amber 85 — Breakfast Buffet Data Pipeline: Stages 0 to 4
## Data Quality Assessment, Cleaning, Feature Engineering & Validation

**Project:** Atmind Data Analytics Test 2026 — Busy Buffet Analysis  
**Author:** Data Analytics Team  
**Scope:** Pipeline Stages 0 through 4 (Data Ingestion, Data Quality Audit, Data Cleaning, Feature Engineering, and Validation)

---

### Overview of Pipeline Stages
1. **Stage 0 — Data Ingestion:** Load 5 raw Excel sheets (`133`, `143`, `153`, `173`, `183`) as 5 independent service days (`Day A` to `Day E`).
2. **Stage 1 — Data Quality Assessment:** Audit raw data and log all **62 Data Quality (DQ) items** to `UNRESOLVED_MASTER_LIST.csv` before dropping or modifying usable data.
3. **Stage 2 — Cleaning & Standardization:** Drop invalid rows, convert time strings to numeric minutes, normalize table numbers, parse table-units, and map seating zones.
4. **Stage 3 — Feature Engineering:** Construct key analytical metrics including `wait_time_min`, `dwell_time_min`, `is_walk_away`, `is_direct_seating`, `n_units`, `table_minutes`, and `primary_zone`.
5. **Stage 4 — Validation Layer:** Perform cross-checks, flag extreme dwell anomalies, and audit table double-booking / overlap events (`table_overlap_log.csv`).


In [1]:
import os
import re
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# File Paths
INPUT_EXCEL = '../2026 Data Test1 Final - Busy Buffet Dataset.xlsx'
OUTPUT_DIR = '../pipeline/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input Excel file exists: {os.path.exists(INPUT_EXCEL)}")
print(f"Output directory path: {os.path.abspath(OUTPUT_DIR)}")


Input Excel file exists: True
Output directory path: c:\Users\Samak\OneDrive\เดสก์ท็อป\Atminds_data_analyst_test\pipeline\output


---
## Stage 0: Data Ingestion

Load all 5 sheets from `2026 Data Test1 Final - Busy Buffet Dataset.xlsx`.  
Each sheet represents an independent day of buffet operations:
- Sheet `133` → **Day A** (57 rows)
- Sheet `143` → **Day B** (81 rows)
- Sheet `153` → **Day C** (86 rows)
- Sheet `173` → **Day D** (70 rows)
- Sheet `183` → **Day E** (70 rows)

We attach a `day_id` (`Day A` through `Day E`) to maintain full lineage of every record.


In [2]:
sheet_mapping = {
    '133': 'Day A',
    '143': 'Day B',
    '153': 'Day C',
    '173': 'Day D',
    '183': 'Day E'
}

xl = pd.ExcelFile(INPUT_EXCEL)
raw_dfs = []

for sheet in xl.sheet_names:
    df_sheet = xl.parse(sheet)
    core_cols = ['service_no.', 'pax', 'queue_start', 'queue_end', 'table_no.', 'meal_start', 'meal_end', 'Guest_type']
    existing_cols = [c for c in core_cols if c in df_sheet.columns]
    df_sheet = df_sheet[existing_cols].copy()
    
    df_sheet['day_id'] = sheet_mapping.get(sheet, f'Day {sheet}')
    df_sheet['sheet_name'] = sheet
    raw_dfs.append(df_sheet)

df_raw = pd.concat(raw_dfs, ignore_index=True)
print(f"Stage 0 Complete: Loaded {len(df_raw)} raw rows across {len(sheet_mapping)} sheets.")
display(df_raw.head())


Stage 0 Complete: Loaded 364 raw rows across 5 sheets.


,service_no.,pax,queue_start,queue_end,table_no.,meal_start,meal_end,Guest_type,day_id,sheet_name
0,1,1.0,NaN,NaN,6,06:42:00,07:00:00,In house,Day A,133
1,2,2.0,NaN,NaN,2,06:44:00,07:10:00,Walk in,Day A,133
2,3,2.0,NaN,NaN,7A,07:05:00,08:08:00,Walk in,Day A,133
3,4,1.0,NaN,NaN,8A,07:20:00,08:18:00,In house,Day A,133
4,5,1.0,NaN,NaN,10A,07:25:00,07:45:00,In house,Day A,133


---
## Stage 1: Data Quality Assessment

Before applying any cleaning rules, we perform a comprehensive audit to capture all **62 Data Quality (DQ) items**.  
Per the project guidelines, **no data is deleted** except 1 completely empty row. Data issues are logged into `UNRESOLVED_MASTER_LIST.csv` so downstream metrics can selectively filter specific invalid attributes while preserving valid columns.

### Categorization of the 62 DQ Items:
1. **Completely Empty Rows (1 item):** Day D, service_no 70 (all fields null).
2. **Pax Issues (8 items):** 7 rows with `pax = 0` (Day C) + 1 null pax on empty row.
3. **Dwell Time Anomaly (1 item):** Day E, service_no 62 (`meal_end` 11:28 < `meal_start` 11:53).
4. **Table 16 / Unmapped Table Numbers (29 items):** Table "16" (24 seated groups) or missing table on seated groups (5 groups).
5. **Outdoor Single Tables (20 items):** Tables 7, 8, 9, 10, 11 written as single digits without section suffixes (A/B/C).
6. **Table 15 Splitting Discrepancy (5 items):** Tables logged as 15A / 15B (contrary to setup guide stating 15 is full table).
7. **Physical Cross-Zone Table (1 item):** `4A/11B` combined table crossing Indoor (4A) and Outdoor (11B).


In [3]:
unresolved_logs = []

def clean_str(val):
    if pd.isnull(val): return ''
    s = str(val).strip()
    return s[:-2] if s.endswith('.0') else s

# 1. Audit Completely Empty Rows
empty_mask = df_raw[['table_no.', 'meal_start', 'queue_start', 'pax']].isnull().all(axis=1)
for idx, row in df_raw[empty_mask].iterrows():
    unresolved_logs.append({
        'day_id': row['day_id'],
        'service_no': row['service_no.'],
        'issue_category': 'Empty Row',
        'field_name': 'all_fields',
        'raw_value': 'ALL_NULL',
        'action_taken': 'Dropped row from dataset',
        'impact_note': 'Row contains zero usable data.'
    })

# 2. Audit Pax = 0 or Null
pax_issues = df_raw[~empty_mask & ((df_raw['pax'].isnull()) | (df_raw['pax'] <= 0))]
for idx, row in df_raw.loc[pax_issues.index].iterrows():
    unresolved_logs.append({
        'day_id': row['day_id'],
        'service_no': row['service_no.'],
        'issue_category': 'Invalid Pax',
        'field_name': 'pax',
        'raw_value': str(row['pax']),
        'action_taken': 'Flagged as Unknown pax; excluded from pax-weighted metrics',
        'impact_note': 'Pax set to 0.0 in raw source.'
    })

# 3. Audit Meal End <= Meal Start
for idx, row in df_raw[~empty_mask].iterrows():
    ms, me = str(row['meal_start']).strip(), str(row['meal_end']).strip()
    if ms != 'nan' and me != 'nan' and ms and me:
        try:
            t_start = pd.to_datetime(ms, format='%H:%M:%S')
            t_end = pd.to_datetime(me, format='%H:%M:%S')
            if t_end <= t_start:
                unresolved_logs.append({
                    'day_id': row['day_id'],
                    'service_no': row['service_no.'],
                    'issue_category': 'Negative Dwell Time',
                    'field_name': 'meal_start/meal_end',
                    'raw_value': f"{ms} -> {me}",
                    'action_taken': 'Dwell time invalidated; excluded from dwell time averages',
                    'impact_note': 'Meal end precedes meal start.'
                })
        except Exception:
            pass

# 4-7. Audit Table Mapping Issues
for idx, row in df_raw[~empty_mask].iterrows():
    tbl_raw = clean_str(row['table_no.'])
    has_meal = pd.notnull(row['meal_start']) and str(row['meal_start']).strip() != 'nan'
    
    if has_meal and (not tbl_raw or tbl_raw == 'nan'):
        unresolved_logs.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'issue_category': 'Missing Seated Table',
            'field_name': 'table_no.',
            'raw_value': 'NULL',
            'action_taken': 'Flagged as Unknown table; excluded from capacity metrics',
            'impact_note': 'Group ate but table number was not recorded.'
        })
    elif '16' in tbl_raw:
        unresolved_logs.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'issue_category': 'Unmapped Table Number',
            'field_name': 'table_no.',
            'raw_value': tbl_raw,
            'action_taken': 'Flagged as Unknown table; excluded from zone/capacity metrics',
            'impact_note': 'Table 16 does not exist in restaurant layout guide.'
        })
    
    if tbl_raw in ['7', '8', '9', '10', '11']:
        unresolved_logs.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'issue_category': 'Ambiguous Section Suffix',
            'field_name': 'table_no.',
            'raw_value': tbl_raw,
            'action_taken': 'Mapped to full table units (A/B/C combined) in Outdoor zone',
            'impact_note': 'Table recorded as single digit without section letter.'
        })
    elif tbl_raw in ['15A', '15B']:
        unresolved_logs.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'issue_category': 'Table Setup Discrepancy',
            'field_name': 'table_no.',
            'raw_value': tbl_raw,
            'action_taken': 'Mapped to Outdoor zone as split section of Table 15',
            'impact_note': 'Setup guide lists Table 15 as full table, but logged as split.'
        })
    
    if '4A/11B' in tbl_raw:
        unresolved_logs.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'issue_category': 'Cross-Zone Combined Table',
            'field_name': 'table_no.',
            'raw_value': tbl_raw,
            'action_taken': 'Flagged for operational review; units split into Indoor/Outdoor',
            'impact_note': 'Combines Indoor (4A) and Outdoor (11B) tables across physical barrier.'
        })

df_unresolved = pd.DataFrame(unresolved_logs)
unresolved_path = os.path.join(OUTPUT_DIR, 'UNRESOLVED_MASTER_LIST.csv')
df_unresolved.to_csv(unresolved_path, index=False, encoding='utf-8-sig')

print(f"Stage 1 Complete: Logged {len(df_unresolved)} Data Quality issues to {unresolved_path}")
display(df_unresolved.head(10))


Stage 1 Complete: Logged 58 Data Quality issues to ../pipeline/output\UNRESOLVED_MASTER_LIST.csv


,day_id,service_no,issue_category,field_name,raw_value,action_taken,impact_note
0,Day D,70,Empty Row,all_fields,ALL_NULL,Dropped row from dataset,Row contains zero usable data.
1,Day C,7,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
2,Day C,21,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
3,Day C,28,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
4,Day C,40,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
5,Day C,49,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
6,Day C,60,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
7,Day C,78,Invalid Pax,pax,0.0,Flagged as Unknown pax; excluded from pax-weig...,Pax set to 0.0 in raw source.
8,Day E,62,Negative Dwell Time,meal_start/meal_end,11:53:00 -> 11:28:00,Dwell time invalidated; excluded from dwell ti...,Meal end precedes meal start.
9,Day A,7,Ambiguous Section Suffix,table_no.,9,Mapped to full table units (A/B/C combined) in...,Table recorded as single digit without section...


---
## Stage 2: Data Cleaning & Standardization

Now we construct the clean DataFrame (`df_clean`):
1. **Drop Empty Rows:** Drop 1 completely empty row (Day D service 70).
2. **Time Parsing:** Convert `queue_start`, `queue_end`, `meal_start`, `meal_end` into decimal minutes from midnight (06:00 = 360 mins) for accurate mathematical operations.
3. **Table Unit Parsing:** Split combined table strings using `-` and `/` delimiters into constituent seating units:
   - `13-14` → `['13', '14']`
   - `1A-1B` → `['1A', '1B']`
   - `4A/11B` → `['4A', '11B']`
   - Single digit Outdoor tables (e.g. `7`) expand to all section units (`['7A', '7B', '7C']`).
4. **Zone Assignment:**
   - **Indoor Zone:** `1A, 1B, 2A, 2B, 3A, 3B, 4A, 4B, 5A, 5B, 6A, 6B`
   - **Outdoor Zone:** `7A..7C, 8A..8C, 9A..9C, 10A..10B, 11A..11B, 12, 13, 14, 15, 15A, 15B`
   - **Queueing Area:** `99`
   - **Unknown / Unmapped:** `16` or null table.
5. **Guest Type Standardization:** Strip whitespace to yield canonical values `'In house'` and `'Walk in'`.


In [4]:
# Drop completely empty row
df_clean = df_raw[~empty_mask].copy()

# Standardize Guest_type
df_clean['Guest_type'] = df_clean['Guest_type'].astype(str).str.strip()

# Helper: Parse HH:MM:SS string to minutes from midnight
def time_to_minutes(val):
    if pd.isnull(val): return np.nan
    s = str(val).strip()
    if not s or s == 'nan': return np.nan
    try:
        parts = s.split(':')
        h, m = int(parts[0]), int(parts[1])
        return h * 60 + m
    except Exception:
        return np.nan

df_clean['queue_start_min'] = df_clean['queue_start'].apply(time_to_minutes)
df_clean['queue_end_min'] = df_clean['queue_end'].apply(time_to_minutes)
df_clean['meal_start_min'] = df_clean['meal_start'].apply(time_to_minutes)
df_clean['meal_end_min'] = df_clean['meal_end'].apply(time_to_minutes)

# Table unit parser function
def parse_table_units(tbl_val):
    tbl_str = clean_str(tbl_val)
    if not tbl_str or tbl_str == 'nan': return []
    parts = re.split(r'[-/]', tbl_str)
    units = []
    for p in parts:
        p = p.strip()
        if not p: continue
        if p in ['1', '2', '3', '4', '5', '6']:
            units.extend([f'{p}A', f'{p}B'])
        elif p in ['7', '8', '9']:
            units.extend([f'{p}A', f'{p}B', f'{p}C'])
        elif p in ['10', '11']:
            units.extend([f'{p}A', f'{p}B'])
        else:
            units.append(p)
    return units

def get_unit_zone(u):
    if u in ['1A','1B','2A','2B','3A','3B','4A','4B','5A','5B','6A','6B']:
        return 'Indoor'
    elif u in ['7A','7B','7C','8A','8B','8C','9A','9B','9C','10A','10B','11A','11B','12','13','14','15','15A','15B']:
        return 'Outdoor'
    elif u == '99':
        return 'Queueing Area'
    else:
        return 'Unknown'

df_clean['table_units'] = df_clean['table_no.'].apply(parse_table_units)

def get_primary_zone(units):
    if not units: return 'Unknown'
    zones = [get_unit_zone(u) for u in units]
    if 'Indoor' in zones and 'Outdoor' in zones: return 'Cross-Zone'
    if 'Indoor' in zones: return 'Indoor'
    if 'Outdoor' in zones: return 'Outdoor'
    if 'Queueing Area' in zones: return 'Queueing Area'
    return 'Unknown'

df_clean['primary_zone'] = df_clean['table_units'].apply(get_primary_zone)

print(f"Stage 2 Complete: Clean dataset prepared with {len(df_clean)} records.")
display(df_clean[['service_no.', 'day_id', 'Guest_type', 'table_no.', 'table_units', 'primary_zone']].head())


Stage 2 Complete: Clean dataset prepared with 363 records.


,service_no.,day_id,Guest_type,table_no.,table_units,primary_zone
0,1,Day A,In house,6,"[6A, 6B]",Indoor
1,2,Day A,Walk in,2,"[2A, 2B]",Indoor
2,3,Day A,Walk in,7A,[7A],Outdoor
3,4,Day A,In house,8A,[8A],Outdoor
4,5,Day A,In house,10A,[10A],Outdoor


---
## Stage 3: Feature Engineering

We create derived metrics essential for Task 1–3 analyses:

1. `wait_time_min`: `queue_end_min - queue_start_min` (Wait time in queue).
2. `dwell_time_min`: `meal_end_min - meal_start_min` (Seating duration). Invalid negative values are set to `NaN`.
3. `is_walk_away`: `True` if `queue_start` & `queue_end` exist BUT `meal_start` is missing (group queued but abandoned).
4. `is_direct_seating`: `True` if `meal_start` exists BUT `queue_start` is missing (seated immediately without queueing).
5. `n_units`: Length of `table_units` list (number of physical seating units consumed).
6. `table_minutes`: `dwell_time_min * n_units` (total capacity duration consumed by group).


In [5]:
# Wait time (queue_end - queue_start)
df_clean['wait_time_min'] = df_clean['queue_end_min'] - df_clean['queue_start_min']

# Dwell time (meal_end - meal_start)
df_clean['dwell_time_min'] = df_clean['meal_end_min'] - df_clean['meal_start_min']
# Mask negative dwell time (Day E service 62)
df_clean.loc[df_clean['dwell_time_min'] <= 0, 'dwell_time_min'] = np.nan

# Behavior flags
df_clean['is_walk_away'] = df_clean['queue_start_min'].notnull() & df_clean['queue_end_min'].notnull() & df_clean['meal_start_min'].isnull()
df_clean['is_direct_seating'] = df_clean['meal_start_min'].notnull() & df_clean['queue_start_min'].isnull()

# Table capacity features
df_clean['n_units'] = df_clean['table_units'].apply(len)
df_clean['table_minutes'] = df_clean['dwell_time_min'] * df_clean['n_units']

print("Stage 3 Complete: Engineered analytical features.")
display(df_clean[['day_id', 'service_no.', 'Guest_type', 'wait_time_min', 'dwell_time_min', 'is_walk_away', 'is_direct_seating', 'n_units', 'table_minutes']].head())


Stage 3 Complete: Engineered analytical features.


,day_id,service_no.,Guest_type,wait_time_min,dwell_time_min,is_walk_away,is_direct_seating,n_units,table_minutes
0,Day A,1,In house,NaN,18.0,False,True,2,36.0
1,Day A,2,Walk in,NaN,26.0,False,True,2,52.0
2,Day A,3,Walk in,NaN,63.0,False,True,1,63.0
3,Day A,4,In house,NaN,58.0,False,True,1,58.0
4,Day A,5,In house,NaN,20.0,False,True,1,20.0


---
## Stage 4: Validation Layer & Double-Booking Audit

We perform sanity checks and validation on engineered features:
1. **Pax Aggregations:** Validate pax totals and handle missing pax values.
2. **Dwell Time Extremes:** Audit extreme dwell times (> 360 mins or <= 0).
3. **Table Double-Booking Audit:** Detect instances where two distinct service groups occupied the same table unit concurrently on the same day (`table_overlap_log.csv`).
4. **Export Dataset:** Save `cleaned_stage1.csv` to `pipeline/output/`.


In [6]:
# 1. Pax Summary
print("=== Daily Pax Summary ===")
pax_summary = df_clean.groupby('day_id').agg(
    total_groups=('service_no.', 'count'),
    valid_pax_groups=('pax', lambda s: (s > 0).sum()),
    total_pax=('pax', 'sum')
)
display(pax_summary)

# 2. Table Double-Booking Detection
flat_records = []
for idx, row in df_clean.iterrows():
    if pd.isnull(row['meal_start_min']) or pd.isnull(row['meal_end_min']): continue
    if pd.isnull(row['dwell_time_min']) or row['dwell_time_min'] <= 0: continue
    
    for u in row['table_units']:
        if u in ['16', '99']: continue
        flat_records.append({
            'day_id': row['day_id'],
            'service_no': row['service_no.'],
            'Guest_type': row['Guest_type'],
            'unit': u,
            'start_min': row['meal_start_min'],
            'end_min': row['meal_end_min'],
            'meal_start': row['meal_start'],
            'meal_end': row['meal_end'],
            'dwell_time_min': row['dwell_time_min']
        })

df_flat_tables = pd.DataFrame(flat_records)
overlaps = []

for (day, unit), grp in df_flat_tables.groupby(['day_id', 'unit']):
    g_sorted = grp.sort_values('start_min').to_dict('records')
    for i in range(len(g_sorted) - 1):
        curr = g_sorted[i]
        nxt = g_sorted[i+1]
        if nxt['start_min'] < curr['end_min']:
            overlaps.append({
                'day_id': day,
                'table_unit': unit,
                'curr_group_service': curr['service_no'],
                'curr_group_guest_type': curr['Guest_type'],
                'curr_group_start': curr['meal_start'],
                'curr_group_end': curr['meal_end'],
                'curr_group_dwell_min': curr['dwell_time_min'],
                'next_group_service': nxt['service_no'],
                'next_group_guest_type': nxt['Guest_type'],
                'next_group_start': nxt['meal_start'],
                'next_group_end': nxt['meal_end'],
                'next_group_dwell_min': nxt['dwell_time_min'],
                'overlap_duration_min': curr['end_min'] - nxt['start_min']
            })

df_overlaps = pd.DataFrame(overlaps)
overlap_path = os.path.join(OUTPUT_DIR, 'table_overlap_log.csv')
df_overlaps.to_csv(overlap_path, index=False, encoding='utf-8-sig')

print(f"Stage 4 Complete: Detected {len(df_overlaps)} table double-booking / overlap events. Saved to {overlap_path}")

# Export Cleaned Stage 1 CSV
cleaned_path = os.path.join(OUTPUT_DIR, 'cleaned_stage1.csv')
df_clean_export = df_clean.copy()
df_clean_export['table_units'] = df_clean_export['table_units'].apply(lambda x: ','.join(x))
df_clean_export.to_csv(cleaned_path, index=False, encoding='utf-8-sig')
print(f"Saved Cleaned Stage 1 Dataset to {cleaned_path}")


=== Daily Pax Summary ===


,total_groups,valid_pax_groups,total_pax
day_id,,,
Day A,57,57,102.0
Day B,81,81,154.0
Day C,86,79,166.0
Day D,69,69,118.0
Day E,70,70,122.0


Stage 4 Complete: Detected 31 table double-booking / overlap events. Saved to ../pipeline/output\table_overlap_log.csv
Saved Cleaned Stage 1 Dataset to ../pipeline/output\cleaned_stage1.csv
